# Model Evaluation and Analysis

Comprehensive evaluation of both VGG16 and ResNet50 models on the test set.

In [1]:
import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Load test dataset
checkpoint = torch.load(
    '../data/processed_datasets.pth',
    map_location='cpu',
    weights_only=False
)
test_dataset = checkpoint['test_dataset']
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"Test samples: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")

Test samples: 6149
Test batches: 193


In [3]:
# Load models
def load_model(model_name, num_classes=102):
    if model_name == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(4096, num_classes)
        model.load_state_dict(torch.load('../models/vgg16_flowers102.pth', map_location=device))
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(2048, num_classes)
        model.load_state_dict(torch.load('../models/resnet50_flowers102.pth', map_location=device))
    
    model = model.to(device)
    model.eval()
    return model

vgg16_model = load_model('vgg16')
resnet50_model = load_model('resnet50')
print("Models loaded successfully!")

Models loaded successfully!


In [ ]:
# Evaluation function
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_preds), np.array(all_labels)

# Evaluate both models
vgg16_preds, test_labels = evaluate_model(vgg16_model, test_loader, device)
resnet50_preds, _ = evaluate_model(resnet50_model, test_loader, device)

print("Evaluation complete!")
print(f"VGG16 Test Accuracy: {accuracy_score(test_labels, vgg16_preds):.4f}")
print(f"ResNet50 Test Accuracy: {accuracy_score(test_labels, resnet50_preds):.4f}")

In [ ]:
# Classification reports (for top 10 classes only due to 102 classes)
top_classes = sorted(set(test_labels))[:10]
mask = np.isin(test_labels, top_classes)

print("VGG16 Classification Report (Top 10 classes):")
print(classification_report(test_labels[mask], vgg16_preds[mask], labels=top_classes))

print("\nResNet50 Classification Report (Top 10 classes):")
print(classification_report(test_labels[mask], resnet50_preds[mask], labels=top_classes))

In [ ]:
# Confusion matrix (for subset of classes)
def plot_confusion_matrix(y_true, y_pred, title, num_classes=10):
    cm = confusion_matrix(y_true, y_pred)
    cm_subset = cm[:num_classes, :num_classes]
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_subset, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'{title} - Confusion Matrix (Top {num_classes} classes)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(test_labels, vgg16_preds, 'VGG16')
plot_confusion_matrix(test_labels, resnet50_preds, 'ResNet50')

In [ ]:
# Top-5 accuracy
def top_k_accuracy(model, test_loader, device, k=5):
    model.eval()
    correct_topk = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            
            _, topk_preds = torch.topk(outputs, k, dim=1)
            correct_topk += sum([1 for i, label in enumerate(labels) if label in topk_preds[i]])
            total += labels.size(0)
    
    return correct_topk / total

vgg16_top5 = top_k_accuracy(vgg16_model, test_loader, device, k=5)
resnet50_top5 = top_k_accuracy(resnet50_model, test_loader, device, k=5)

print(f"VGG16 Top-5 Accuracy: {vgg16_top5:.4f}")
print(f"ResNet50 Top-5 Accuracy: {resnet50_top5:.4f}")

In [ ]:
# Save evaluation results
results = {
    'vgg16': {
        'top1_accuracy': accuracy_score(test_labels, vgg16_preds),
        'top5_accuracy': vgg16_top5,
        'predictions': vgg16_preds
    },
    'resnet50': {
        'top1_accuracy': accuracy_score(test_labels, resnet50_preds),
        'top5_accuracy': resnet50_top5,
        'predictions': resnet50_preds
    },
    'test_labels': test_labels
}

torch.save(results, '../models/evaluation_results.pth')
print("Evaluation results saved successfully!")